# 🖋️ OmniFile Arabic OCR — النسخة الموحدة النهائية

**المطور:** Dr. Abdulmalek Tamer Al-husseini | Homs, Syria  
**GitHub:** [OmniFile_Processor](https://github.com/DrAbdulmalek/OmniFile_Processor)

---

## ✨ المميزات المدمجة

| الميزة | المصدر |
|--------|--------|
| 🔤 OCR بثلاث محركات (TrOCR + EasyOCR + Tesseract) | جميع النسخ |
| 🧠 قاموس تصحيح ذكي بـ CorrectionRule + معايرة تلقائية | v5 Enhanced |
| 📄 معالجة صفحة بصفحة (ذاكرة منخفضة) | v5 DeepSeek |
| 🔒 FileLock للأجهزة المتعددة | v5 Enhanced |
| 📊 WER/CER مقاييس أداء + رسوم بيانية | ZAI Colab |
| 📚 دليل دراسة + مخططات Mermaid + HTML RTL | v5 DeepSeek |
| 🔄 التراجع (Undo) + نسخ النص الأصلي | v5 Enhanced |
| 📑 تصدير شامل (CSV + XLSX + JSONL + PDF + TXT) | Gradio Pro |
| 🖼️ تصحيح الميل (Deskew) + معالجة الصور | Monitoring |
| 🗃️ SmartMigrator — استيراد من 8 مشاريع قديمة | v5 DeepSeek |
| 🌍 قاموس ثنائي اللغة (عربي-إنجليزي) | Claude |
| 🧪 Active Learning + LoRA Fine-tuning | جميع النسخ |
| 📈 مراقبة شاملة (JSONL + Health Snapshots) | Monitoring |
| 🎯 Gradio UI بـ 8 تبويبات | موحد |

---

> **للاستخدام على Google Colab:** شغّل الخلايا بالترتيب من الأعلى إلى الأسفل.
> يتم تثبيت المكتبات تلقائياً في الخلية الأولى.


In [ ]:
%%bash# ========================================# الخلية 1: تثبيت المكتبات — شغّلها مرة واحدة فقط# ========================================apt-get update -qq && apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-ara tesseract-ocr-eng > /dev/null 2>&1echo "✅ System packages installed"pip install -q \    easyocr \    transformers \    torch \    torchvision \    accelerate \    peft \    pytesseract \    Pillow \    opencv-python-headless \    pdf2image \    pandas \    openpyxl \    fpdf2 \    pyspellchecker \    langdetect \    ar-corrector \    arabic-reshaper \    python-bidi \    jiwer \    albumentations \    tensorboard \    matplotlib \    seaborn \    gradio \    nlpaug \    aubmindlab-bert-base-arabertv2 \    huggingface_hub \    > /dev/null 2>&1echo "✅ All Python packages installed successfully"

In [ ]:
# ========================================# الخلية 2: الاستيرادات# ========================================import os, sys, json, csv, time, logging, sqlite3, shutil, hashlib, threadingfrom datetime import datetimefrom dataclasses import dataclass, field, asdictfrom pathlib import Pathfrom typing import Optional, Dict, List, Tuple, Anyfrom contextlib import contextmanagerfrom collections import defaultdict, Counterfrom concurrent.futures import ThreadPoolExecutor, as_completedimport cv2import numpy as npimport pandas as pdfrom PIL import Imageimport easyocrimport pytesseract# Spell checkingfrom pyspellchecker import SpellChecker as EnSpellCheckerfrom ar_corrector.corrector import Corrector as ArCorrector# Arabic textimport arabic_reshaperfrom bidi.algorithm import get_display# ML / Transformersimport torchfrom transformers import (    VisionEncoderDecoderModel,     TrOCRProcessor,     AutoTokenizer,     AutoModelForSeq2SeqLM,    AutoModelForCausalLM,    Trainer, TrainingArguments,    EarlyStoppingCallback,    default_data_collator)from peft import LoraConfig, get_peft_model, TaskType# Metricsfrom jiwer import wer, cer# Gradioimport gradio as gr# PDFfrom pdf2image import convert_from_path# PDF Exportfrom fpdf import FPDF# Google Colab detectionIN_COLAB = 'google.colab' in sys.modulesif IN_COLAB:    from google.colab import drive, userdata, files    drive.mount('/content/drive')    print("✅ Google Drive mounted")else:    print("ℹ️ Not running in Colab")print("✅ All imports successful")

In [ ]:
# ========================================# الخلية 3: الإعدادات المركزية# ========================================@dataclassclass Config:    # Paths    base_dir: str = "/content/drive/MyDrive/OmniFile_ArabicOCR" if IN_COLAB else "./OmniFile_ArabicOCR"    db_path: str = ""  # auto-set in __post_init__    feedback_csv: str = ""  # auto-set    checkpoint_path: str = ""  # auto-set    events_log: str = ""  # auto-set        # OCR Models    trocr_model_name: str = "David-Magdy/TR_OCR_LARGE"    easyocr_langs: list = field(default_factory=lambda: ['ar', 'en'])    use_tesseract: bool = True    tesseract_langs: str = "ara+eng"        # OCR Parameters    trocr_beam_size: int = 5    trocr_num_return_sequences: int = 3  # top-k predictions    easy_conf_threshold: float = 0.80    trocr_conf_threshold: float = 0.50    ensemble_mode: str = "best_confidence"  # best_confidence, voting, all_candidates        # Memory Management    memory_mode: str = "auto"  # low, high, auto    use_fp16: bool = True    trocr_batch_size: int = 4    max_pages_in_memory: int = 5        # Processing    dpi: int = 300    deskew_enabled: bool = True    clahe_enabled: bool = True    denoise_enabled: bool = True    adaptive_threshold: bool = True        # Spell Checking    enable_spell_check: bool = True    min_word_length: int = 2    custom_dict_path: str = ""        # Dictionary / CorrectionRule thresholds    dict_conf_low: float = 0.3    dict_conf_mid: float = 0.7    dict_usage_high: int = 10    dict_usage_mid: int = 3    dict_days_warning: int = 30    dict_days_critical: int = 90        # Active Learning    al_min_corrections: int = 20    al_epochs_trigger: int = 3    al_low_conf_threshold: float = 0.6        # LoRA Fine-tuning    lora_r: int = 16    lora_alpha: int = 32    lora_dropout: float = 0.1    lora_target_modules: list = field(default_factory=lambda: ["query", "value"])    ft_epochs: int = 3    ft_batch_size: int = 4    ft_learning_rate: float = 5e-5    ft_max_samples: int = 500    use_albumentations: bool = True        # Export    auto_export_after_run: bool = True    export_csv: bool = True    export_xlsx: bool = True    export_jsonl: bool = True    export_txt: bool = True    export_pdf_report: bool = False    export_training_data: bool = True        # Gradio    gradio_share: bool = True    gradio_server_name: str = "0.0.0.0"    gradio_server_port: int = 0    gradio_debug: bool = False        # Migration    old_project_names: list = field(default_factory=lambda: [        "HandwrittenOCR",        "Arabic_OCR_v5",        "Arabic_OCR_Ultimate",         "OmniFile_v500",        "handwriting-ocr-arabic",        "arabic_ocr_project",        "OCR_Correction",        "Arabic_Handwriting_OCR"    ])        def __post_init__(self):        os.makedirs(self.base_dir, exist_ok=True)        self.db_path = os.path.join(self.base_dir, "handwriting.db")        self.feedback_csv = os.path.join(self.base_dir, "feedback.csv")        self.checkpoint_path = os.path.join(self.base_dir, "checkpoint.json")        self.events_log = os.path.join(self.base_dir, "ocr_events.jsonl")cfg = Config()print(f"✅ Configuration loaded")print(f"   Base dir: {cfg.base_dir}")print(f"   DB: {cfg.db_path}")print(f"   Memory mode: {cfg.memory_mode}")print(f"   FP16: {cfg.use_fp16}")

In [ ]:
# ========================================# الخلية 4: نظام التسجيل والمراقبة# ========================================def setup_logging(config: Config):    """Setup rotating file logger + JSONL events logger."""    log_dir = os.path.join(config.base_dir, "logs")    os.makedirs(log_dir, exist_ok=True)        logger = logging.getLogger("OmniFile_OCR")    logger.setLevel(logging.DEBUG)        # Rotating file handler (2MB, 5 backups)    fh = logging.handlers.RotatingFileHandler(        os.path.join(log_dir, "omnifile.log"),        maxBytes=2*1024*1024,        backupCount=5,        encoding='utf-8'    )    fh.setLevel(logging.DEBUG)    fh.setFormatter(logging.Formatter(        '%(asctime)s | %(levelname)-8s | %(message)s',        datefmt='%Y-%m-%d %H:%M:%S'    ))    logger.addHandler(fh)        # Console handler    ch = logging.StreamHandler()    ch.setLevel(logging.INFO)    ch.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))    logger.addHandler(ch)        return loggerimport logging.handlerslogger = setup_logging(cfg)def log_event(event_type: str, details: dict = None):    """Log structured event to JSONL events file."""    event = {        "timestamp": datetime.now().isoformat(),        "type": event_type,        **(details or {})    }    try:        with open(cfg.events_log, 'a', encoding='utf-8') as f:            f.write(json.dumps(event, ensure_ascii=False) + '\n')    except Exception as e:        logger.warning(f"Failed to log event: {e}")def write_health_snapshot():    """Write system health snapshot to JSON."""    import platform    snapshot = {        "timestamp": datetime.now().isoformat(),        "system": platform.platform(),        "python_version": platform.python_version(),        "cuda_available": torch.cuda.is_available() if 'torch' in dir() else False,        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",        "gpu_memory_total": torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0,        "gpu_memory_allocated": torch.cuda.memory_allocated(0) / 1e9 if torch.cuda.is_available() else 0,        "disk_usage": shutil.disk_usage(cfg.base_dir),    }    path = os.path.join(cfg.base_dir, "health_snapshot.json")    with open(path, 'w', encoding='utf-8') as f:        json.dump(snapshot, f, indent=2, ensure_ascii=False, default=str)    return snapshotprint("✅ Logging and monitoring system initialized")

In [ ]:
# ========================================# الخلية 5: قفل الملفات للأجهزة المتعددة# ========================================import fcntlclass FileLock:    """Cross-platform file lock for multi-device safety on Google Drive."""        def __init__(self, lockfile: str, timeout: float = 30.0):        self.lockfile = lockfile        self.timeout = timeout        self._fd = None        def __enter__(self):        os.makedirs(os.path.dirname(self.lockfile) or '.', exist_ok=True)        self._fd = open(self.lockfile, 'w')        start = time.time()        while True:            try:                fcntl.flock(self._fd, fcntl.LOCK_EX | fcntl.LOCK_NB)                self._fd.write(f"{os.getpid()}\n{datetime.now().isoformat()}\n")                self._fd.flush()                return self            except (IOError, OSError):                if time.time() - start > self.timeout:                    raise TimeoutError(f"Cannot acquire lock: {self.lockfile}")                time.sleep(0.5)        def __exit__(self, *args):        if self._fd:            try:                fcntl.flock(self._fd, fcntl.LOCK_UN)            except Exception:                pass            self._fd.close()_db_lock = FileLock(os.path.join(cfg.base_dir, ".db.lock"))print("✅ FileLock initialized")

In [ ]:
# ========================================# الخلية 6: قاعدة البيانات (SQLite)# ========================================SCHEMA_V3 = """CREATE TABLE IF NOT EXISTS handwriting_data (    id INTEGER PRIMARY KEY AUTOINCREMENT,    page_num INTEGER NOT NULL,    word_bbox TEXT,    raw_text TEXT NOT NULL,    corrected_text TEXT,    confidence REAL,    ocr_engine TEXT,    top_predictions TEXT,    status TEXT DEFAULT 'pending',    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,    run_id TEXT);CREATE TABLE IF NOT EXISTS processing_runs (    run_id TEXT PRIMARY KEY,    started_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,    completed_at TIMESTAMP,    total_pages INTEGER DEFAULT 0,    total_words INTEGER DEFAULT 0,    status TEXT DEFAULT 'running',    notes TEXT);CREATE TABLE IF NOT EXISTS review_events (    id INTEGER PRIMARY KEY AUTOINCREMENT,    run_id TEXT,    word_id INTEGER,    action TEXT,    old_value TEXT,    new_value TEXT,    reviewer TEXT DEFAULT 'user',    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);CREATE INDEX IF NOT EXISTS idx_status ON handwriting_data(status);CREATE INDEX IF NOT EXISTS idx_page ON handwriting_data(page_num);CREATE INDEX IF NOT EXISTS idx_confidence ON handwriting_data(confidence);CREATE INDEX IF NOT EXISTS idx_run ON handwriting_data(run_id);CREATE INDEX IF NOT EXISTS idx_review_run ON review_events(run_id);CREATE INDEX IF NOT EXISTS idx_review_word ON review_events(word_id);"""class HandwritingDB:    def __init__(self, db_path: str):        self.db_path = db_path        self._conn = None        self._init_db()        def _init_db(self):        with FileLock(self.db_path + ".lock"):            conn = sqlite3.connect(self.db_path, timeout=30)            conn.executescript("PRAGMA journal_mode=WAL; PRAGMA synchronous=NORMAL;")            conn.executescript(SCHEMA_V3)            # Add missing columns if needed (migration)            try:                conn.execute("ALTER TABLE handwriting_data ADD COLUMN top_predictions TEXT")            except sqlite3.OperationalError:                pass            try:                conn.execute("ALTER TABLE handwriting_data ADD COLUMN run_id TEXT")            except sqlite3.OperationalError:                pass            conn.commit()            conn.close()        @property    def conn(self):        if self._conn is None:            self._conn = sqlite3.connect(self.db_path, timeout=30)            self._conn.row_factory = sqlite3.Row        return self._conn        def close(self):        if self._conn:            self._conn.close()            self._conn = None        def insert_word(self, page_num, raw_text, confidence, ocr_engine,                     word_bbox=None, top_predictions=None, run_id=None):        sql = """INSERT INTO handwriting_data                  (page_num, word_bbox, raw_text, confidence, ocr_engine, top_predictions, run_id)                 VALUES (?, ?, ?, ?, ?, ?, ?)"""        with _db_lock:            self.conn.execute(sql, (page_num, word_bbox, raw_text, confidence,                                    ocr_engine, json.dumps(top_predictions) if top_predictions else None, run_id))            self.conn.commit()        return self.conn.execute("SELECT last_insert_rowid()").fetchone()[0]        def get_words(self, page_num=None, status=None, run_id=None, limit=None):        sql = "SELECT * FROM handwriting_data WHERE 1=1"        params = []        if page_num is not None:            sql += " AND page_num = ?"            params.append(page_num)        if status is not None:            sql += " AND status = ?"            params.append(status)        if run_id is not None:            sql += " AND run_id = ?"            params.append(run_id)        sql += " ORDER BY page_num, id"        if limit:            sql += f" LIMIT {limit}"        return [dict(r) for r in self.conn.execute(sql, params).fetchall()]        def update_word(self, word_id, corrected_text=None, status=None, confidence=None):        sets = []        params = []        if corrected_text is not None:            sets.append("corrected_text = ?")            params.append(corrected_text)        if status is not None:            sets.append("status = ?")            params.append(status)        if confidence is not None:            sets.append("confidence = ?")            params.append(confidence)        sets.append("updated_at = CURRENT_TIMESTAMP")        params.append(word_id)        with _db_lock:            self.conn.execute(f"UPDATE handwriting_data SET {', '.join(sets)} WHERE id = ?", params)            self.conn.commit()        def delete_word(self, word_id):        with _db_lock:            self.conn.execute("DELETE FROM handwriting_data WHERE id = ?", (word_id,))            self.conn.commit()        def log_review(self, run_id, word_id, action, old_value=None, new_value=None, reviewer='user'):        with _db_lock:            self.conn.execute(                "INSERT INTO review_events (run_id, word_id, action, old_value, new_value, reviewer) VALUES (?,?,?,?,?,?)",                (run_id, word_id, action, old_value, new_value, reviewer)            )            self.conn.commit()        def count_by_status(self, run_id=None):        sql = "SELECT status, COUNT(*) as cnt FROM handwriting_data"        params = []        if run_id:            sql += " WHERE run_id = ?"            params.append(run_id)        sql += " GROUP BY status"        return {r['status']: r['cnt'] for r in self.conn.execute(sql, params).fetchall()}        def start_run(self, run_id, notes=""):        with _db_lock:            self.conn.execute(                "INSERT OR REPLACE INTO processing_runs (run_id, started_at, status, notes) VALUES (?, CURRENT_TIMESTAMP, 'running', ?)",                (run_id, notes)            )            self.conn.commit()        def complete_run(self, run_id, total_pages=0, total_words=0, status='completed'):        with _db_lock:            self.conn.execute(                "UPDATE processing_runs SET completed_at=CURRENT_TIMESTAMP, total_pages=?, total_words=?, status=? WHERE run_id=?",                (total_pages, total_words, status, run_id)            )            self.conn.commit()        def get_run_history(self, limit=10):        return [dict(r) for r in self.conn.execute(            "SELECT * FROM processing_runs ORDER BY started_at DESC LIMIT ?", (limit,)        ).fetchall()]        def get_low_confidence(self, threshold=0.6, run_id=None, limit=50):        sql = "SELECT * FROM handwriting_data WHERE confidence < ? AND status = 'pending'"        params = [threshold]        if run_id:            sql += " AND run_id = ?"            params.append(run_id)        sql += " ORDER BY confidence ASC LIMIT ?"        params.append(limit)        return [dict(r) for r in self.conn.execute(sql, params).fetchall()]        def get_verified(self, run_id=None, limit=None):        return self.get_words(status='verified', run_id=run_id, limit=limit)        def get_review_events(self, run_id=None, limit=50):        sql = "SELECT * FROM review_events"        params = []        if run_id:            sql += " WHERE run_id = ?"            params.append(run_id)        sql += " ORDER BY created_at DESC LIMIT ?"        params.append(limit)        return [dict(r) for r in self.conn.execute(sql, params).fetchall()]        def delete_run_data(self, run_id):        with _db_lock:            self.conn.execute("DELETE FROM handwriting_data WHERE run_id = ?", (run_id,))            self.conn.execute("DELETE FROM review_events WHERE run_id = ?", (run_id,))            self.conn.commit()db = HandwritingDB(cfg.db_path)print(f"✅ Database initialized at {cfg.db_path}")log_event("db_init", {"path": cfg.db_path})

In [ ]:
# ========================================# الخلية 7: استيراد البيانات من المشاريع القديمة# ========================================class SmartMigrator:    """Scan, import, and optionally delete data from old project folders."""        def __init__(self, config: Config, database: HandwritingDB):        self.cfg = config        self.db = database        self.found_projects = {}        def scan(self) -> dict:        """Scan Drive for old project folders."""        drive_root = os.path.dirname(self.cfg.base_dir)        for name in self.cfg.old_project_names:            path = os.path.join(drive_root, name)            if os.path.isdir(path):                db_path = os.path.join(path, "handwriting.db")                csv_path = os.path.join(path, "feedback.csv")                json_path = os.path.join(path, "correction_dict.json")                info = {"path": path, "has_db": False, "has_csv": False, "has_json": False}                if os.path.exists(db_path):                    info["has_db"] = True                    info["db_size"] = os.path.getsize(db_path)                if os.path.exists(csv_path):                    info["has_csv"] = True                    info["csv_rows"] = sum(1 for _ in open(csv_path)) - 1                if os.path.exists(json_path):                    info["has_json"] = True                self.found_projects[name] = info        return self.found_projects        def migrate(self, delete_after: bool = False) -> dict:        """Import data from all found projects."""        report = {"imported": 0, "errors": [], "details": {}}        for name, info in self.found_projects.items():            detail = {"db_words": 0, "csv_rows": 0, "json_rules": 0}            path = info["path"]                        # Import DB words            if info["has_db"]:                try:                    old_db = sqlite3.connect(os.path.join(path, "handwriting.db"))                    rows = old_db.execute(                        "SELECT page_num, raw_text, corrected_text, confidence, status FROM handwriting_data"                    ).fetchall()                    for row in rows:                        self.db.insert_word(                            page_num=row[0] or 0,                            raw_text=row[1],                            confidence=row[3] or 0.0,                            ocr_engine='migrated',                            run_id=f"migrated_{name}"                        )                        if row[2]:  # has correction                            # update with corrected text                            wid = self.db.conn.execute("SELECT last_insert_rowid()").fetchone()[0]                            self.db.update_word(wid, corrected_text=row[2], status=row[4] or 'verified')                    detail["db_words"] = len(rows)                    old_db.close()                except Exception as e:                    report["errors"].append(f"{name} DB: {e}")                        # Import feedback CSV            if info["has_csv"]:                try:                    with open(os.path.join(path, "feedback.csv"), 'r', encoding='utf-8') as f:                        reader = csv.DictReader(f)                        count = 0                        for row in reader:                            count += 1                    detail["csv_rows"] = count                except Exception as e:                    report["errors"].append(f"{name} CSV: {e}")                        # Import correction dict JSON            if info["has_json"]:                try:                    with open(os.path.join(path, "correction_dict.json"), 'r', encoding='utf-8') as f:                        data = json.load(f)                    detail["json_rules"] = len(data)                except Exception as e:                    report["errors"].append(f"{name} JSON: {e}")                        report["imported"] += detail["db_words"]            report["details"][name] = detail                        if delete_after and detail["db_words"] > 0:                backup = path + f"_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"                shutil.copytree(path, backup)                shutil.rmtree(path)                detail["deleted"] = True                detail["backup"] = backup                log_event("migration", report)        return reportmigrator = SmartMigrator(cfg, db)print("✅ SmartMigrator ready")print(f"   Old project patterns: {len(cfg.old_project_names)}")

In [ ]:
# ========================================# الخلية 8: معالجة الصور (Preprocessing)# ========================================def deskew_image(img: np.ndarray) -> np.ndarray:    """Correct skew/rotation using minAreaRect."""    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]    coords = np.column_stack(np.where(thresh > 0))    if len(coords) == 0:        return img    angle = cv2.minAreaRect(coords)[-1]    if angle < -45:        angle = -(90 + angle)    else:        angle = -angle    if abs(angle) < 0.5:  # negligible angle        return img    h, w = img.shape[:2]    center = (w // 2, h // 2)    M = cv2.getRotationMatrix2D(center, angle, 1.0)    rotated = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)    return rotateddef preprocess_image(image: np.ndarray, config: Config = None) -> np.ndarray:    """Full preprocessing pipeline: deskew → CLAHE → denoise → threshold."""    if config is None:        config = cfg        # Deskew    if config.deskew_enabled:        image = deskew_image(image)        # Convert to grayscale    if len(image.shape) == 3:        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)    else:        gray = image.copy()        # CLAHE (Contrast Limited Adaptive Histogram Equalization)    if config.clahe_enabled:        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))        gray = clahe.apply(gray)        # Denoising    if config.denoise_enabled:        gray = cv2.fastNlMeansDenoising(gray, None, h=10, templateWindowSize=7, searchWindowSize=21)        # Adaptive threshold    if config.adaptive_threshold:        binary = cv2.adaptiveThreshold(            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,            cv2.THRESH_BINARY, 11, 2        )    else:        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)        # Dilation to connect broken characters    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))    binary = cv2.dilate(binary, kernel, iterations=1)        return binarydef crop_safe(img: np.ndarray, x: int, y: int, w: int, h: int) -> np.ndarray:    """Safely crop image region, handling boundary cases."""    H, W = img.shape[:2]    x1, y1 = max(0, x), max(0, y)    x2, y2 = min(W, x + w), min(H, y + h)    if x2 <= x1 or y2 <= y1:        return img    return img[y1:y2, x1:x2]print("✅ Image preprocessing functions loaded")

In [ ]:
# ========================================# الخلية 9: محرك OCR (TrOCR + EasyOCR + Tesseract)# ========================================class OCREngine:    """Multi-engine OCR with ensemble strategies and memory management."""        def __init__(self, config: Config):        self.cfg = config        self.trocr_model = None        self.trocr_processor = None        self.easy_reader = None        self.tesseract_available = False        self._device = "cuda" if torch.cuda.is_available() else "cpu"        self._load_models()        def _load_models(self):        """Load all OCR models with memory optimization."""        # Auto-detect memory mode        if self.cfg.memory_mode == "auto":            if torch.cuda.is_available():                gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9                self.cfg.memory_mode = "high" if gpu_mem >= 12 else "low"            else:                self.cfg.memory_mode = "low"            print(f"  Auto-detected memory mode: {self.cfg.memory_mode}")                # Clear HuggingFace cache to save space        cache_dir = os.path.expanduser("~/.cache/huggingface/hub")        if os.path.exists(cache_dir) and IN_COLAB:            shutil.rmtree(cache_dir, ignore_errors=True)                # TrOCR        print("  Loading TrOCR...")        self.trocr_model = VisionEncoderDecoderModel.from_pretrained(            self.cfg.trocr_model_name        ).to(self._device)        if self.cfg.use_fp16 and self._device == "cuda":            self.trocr_model.half()        self.trocr_model.eval()                self.trocr_processor = TrOCRProcessor.from_pretrained(self.cfg.trocr_model_name)                # EasyOCR        print("  Loading EasyOCR...")        gpu_mode = self.cfg.memory_mode == "high"        self.easy_reader = easyocr.Reader(            self.cfg.easyocr_langs,            gpu=gpu_mode,            verbose=False        )                # Tesseract        if self.cfg.use_tesseract:            try:                pytesseract.get_tesseract_version()                self.tesseract_available = True                print("  Tesseract: available")            except Exception:                print("  Tesseract: not available")                # Optimize CPU threads for low memory        if self._device == "cpu":            torch.set_num_threads(2)                print(f"  Device: {self._device} | FP16: {self.cfg.use_fp16}")        def predict_trocr(self, image):        """Run TrOCR with top-k beam search. Returns list of (text, confidence) tuples."""        try:            pil_img = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)) if len(image.shape) == 3 else Image.fromarray(image)            pixel_values = self.trocr_processor(pil_img, return_tensors="pt").pixel_values.to(self._device)                        if self.cfg.use_fp16 and self._device == "cuda":                pixel_values = pixel_values.half()                        with torch.no_grad():                generated = self.trocr_model.generate(                    pixel_values,                    num_beams=self.cfg.trocr_beam_size,                    num_return_sequences=self.cfg.trocr_num_return_sequences,                    early_stopping=True                )                        results = []            for seq in generated:                text = self.trocr_processor.tokenizer.decode(seq, skip_special_tokens=True).strip()                # Estimate confidence from beam score                try:                    score = self.trocr_model(pixel_values, labels=seq).loss.item()                    confidence = max(0, 1 - score)                except Exception:                    confidence = 0.8                results.append((text, round(confidence, 4)))                        return results        except Exception as e:            logger.error(f"TrOCR prediction failed: {e}")            return [("", 0.0)]        def predict_easyocr(self, image):        """Run EasyOCR. Returns list of (text, confidence) tuples."""        try:            results = self.easy_reader.readtext(image)            return [(text.strip(), float(conf)) for (bbox, text, conf) in results]        except Exception as e:            logger.error(f"EasyOCR prediction failed: {e}")            return [("", 0.0)]        def predict_tesseract(self, image):        """Run Tesseract OCR. Returns list of (text, confidence) tuples."""        if not self.tesseract_available:            return [("", 0.0)]        try:            data = pytesseract.image_to_data(                image, lang=self.cfg.tesseract_langs,                output_type=pytesseract.Output.DICT, config="--psm 7"            )            results = []            for i in range(len(data["text"])):                text = data["text"][i].strip()                conf = int(data["conf"][i]) if data["conf"][i] != "-1" else 0                if text and conf > 0:                    results.append((text, conf / 100.0))            return results        except Exception as e:            logger.error(f"Tesseract prediction failed: {e}")            return [("", 0.0)]        def ensemble(self, image):        """Run all engines and merge results. Returns best result + all candidates."""        # Run all engines        trocr_results = self.predict_trocr(image)        easy_results = self.predict_easyocr(image)        tess_results = self.predict_tesseract(image)                # If EasyOCR is very confident, prefer it        if (easy_results and easy_results[0][1] >= self.cfg.easy_conf_threshold             and (not trocr_results[0][0] or trocr_results[0][1] < easy_results[0][1])):            return self._format_result(easy_results, trocr_results, tess_results, "easyocr")                all_candidates = {}        for text, conf in trocr_results:            if text:                all_candidates[text] = conf        for text, conf in easy_results:            if text:                all_candidates[text] = max(all_candidates.get(text, 0), conf)        for text, conf in tess_results:            if text:                all_candidates[text] = max(all_candidates.get(text, 0), conf)                if not all_candidates:            return {"text": "", "confidence": 0.0, "engine": "none", "all_candidates": {},                    "trocr_predictions": [], "easyocr_predictions": [], "tesseract_predictions": []}                best_text = max(all_candidates, key=all_candidates.get)        best_conf = all_candidates[best_text]                engine = "trocr"        if easy_results and easy_results[0][0] == best_text and easy_results[0][1] >= best_conf:            engine = "easyocr"        elif tess_results and tess_results[0][0] == best_text:            engine = "tesseract"                return {            "text": best_text, "confidence": best_conf, "engine": engine,            "all_candidates": all_candidates,            "trocr_predictions": trocr_results,            "easyocr_predictions": easy_results,            "tesseract_predictions": tess_results        }        def _format_result(self, primary, trocr, tess, engine):        all_cands = {}        for t, c in primary:            if t: all_cands[t] = c        for t, c in trocr:            if t: all_cands[t] = max(all_cands.get(t, 0), c)        for t, c in tess:            if t: all_cands[t] = max(all_cands.get(t, 0), c)                best_text = primary[0][0] if primary else ""        best_conf = primary[0][1] if primary else 0.0                return {            "text": best_text, "confidence": best_conf, "engine": engine,            "all_candidates": all_cands,            "trocr_predictions": trocr,            "easyocr_predictions": primary,            "tesseract_predictions": tess        }# Initialize OCR engineprint("Loading OCR engines...")ocr_engine = OCREngine(cfg)print("OCR engine initialized (TrOCR + EasyOCR + Tesseract)")log_event("ocr_init", {"device": ocr_engine._device, "fp16": cfg.use_fp16})

In [ ]:
# ========================================# الخلية 10: التصحيح الإملائي + القاموس الذكي# ========================================@dataclassclass CorrectionRule:    """Metadata-rich correction rule with confidence scoring."""    original: str    correction: str    confidence: float = 0.0    usage_count: int = 0    votes_up: int = 0    votes_down: int = 0    contexts: list = field(default_factory=list)    created_at: str = ""    last_used: str = ""    reviewer: str = ""    status: str = "pending"  # pending, approved, rejected    flagged: bool = False        def net_votes(self):        return self.votes_up - self.votes_downdef build_correction_dict(feedback_csv, min_freq=2):    """Build correction dictionary from feedback CSV."""    corrections = {}    if not os.path.exists(feedback_csv):        return corrections        counter = defaultdict(lambda: defaultdict(int))    with open(feedback_csv, "r", encoding="utf-8") as f:        reader = csv.DictReader(f)        for row in reader:            orig = row.get("raw_text", "").strip()            corr = row.get("corrected_text", "").strip()            if orig and corr and orig != corr:                counter[orig][corr] += 1        for orig, corr_counts in counter.items():        best = max(corr_counts, key=corr_counts.get)        if corr_counts[best] >= min_freq:            rule = CorrectionRule(                original=orig, correction=best,                confidence=round(corr_counts[best] / sum(corr_counts.values()), 3),                usage_count=corr_counts[best],                created_at=datetime.now().isoformat(),                status="approved"            )            corrections[orig] = rule    return correctionsdef load_correction_rules(config):    """Load correction rules from JSON or build from feedback + verified DB data."""    json_path = os.path.join(config.base_dir, "correction_rules.json")    if os.path.exists(json_path):        try:            with open(json_path, "r", encoding="utf-8") as f:                data = json.load(f)            return {k: CorrectionRule(**v) for k, v in data.items()}        except Exception as e:            logger.warning(f"Failed to load correction rules: {e}")        rules = build_correction_dict(config.feedback_csv)        # Also build from verified DB words    verified = db.get_verified()    for w in verified:        if w["raw_text"] and w["corrected_text"] and w["raw_text"] != w["corrected_text"]:            orig = w["raw_text"].strip()            corr = w["corrected_text"].strip()            if orig not in rules:                rules[orig] = CorrectionRule(                    original=orig, correction=corr,                    confidence=round(w.get("confidence", 0.8), 3),                    status="approved",                    created_at=w.get("created_at", datetime.now().isoformat())                )    return rulesdef calculate_rule_indicator(rule):    """Visual priority indicator for a correction rule."""    if rule.status == "rejected":        return "X"    if rule.status == "pending" and rule.usage_count == 0:        return "hourglass"    if rule.flagged:        return "flag"        score = 0    score += min(rule.confidence * 10, 5)    score += min(rule.usage_count, 5)    score += min(rule.net_votes(), 3)        if score >= 8:        return "green"    elif score >= 4:        return "yellow"    else:        return "red"def auto_calibrate_dict_thresholds(rules):    """Statistically calibrate thresholds."""    if not rules:        return {}    confs = [r.confidence for r in rules.values() if r.status == "approved"]    usages = [r.usage_count for r in rules.values() if r.status == "approved"]    if not confs:        return {}    import statistics    return {        "conf_low": round(statistics.mean(confs) - statistics.stdev(confs), 3) if len(confs) > 1 else 0.3,        "conf_mid": round(statistics.median(confs), 3),        "usage_high": round(statistics.mean(usages) + statistics.stdev(usages)) if len(usages) > 1 else 10,        "usage_mid": round(statistics.median(usages)) if usages else 3,    }def apply_corrections(text, rules):    """Apply correction dictionary to text. Returns (corrected_text, applied_rules)."""    words = text.split()    applied = []    for i, word in enumerate(words):        clean = word.strip(".,;:!?()[]{}\'\"-")        if clean in rules:            rule = rules[clean]            if rule.status in ("approved", "pending"):                words[i] = words[i].replace(clean, rule.correction)                rule.usage_count += 1                rule.last_used = datetime.now().isoformat()                applied.append(rule)    return " ".join(words), applied# Initialize spell checkersar_corrector = ArCorrector()en_spell = EnSpellChecker(language="en")def detect_language_safe(text):    """Detect language with fallback."""    try:        from langdetect import detect        return detect(text)    except Exception:        arabic_count = sum(1 for c in text if "\u0600" <= c <= "\u06FF")        return "ar" if arabic_count > len(text) * 0.3 else "en"def correct_spelling(text):    """Correct spelling using ar-corrector or pyspellchecker."""    if not cfg.enable_spell_check or not text.strip():        return text    lang = detect_language_safe(text)    if lang == "ar":        try:            return ar_corrector.contextual_correct(text)        except Exception:            return text    else:        try:            words = text.split()            corrected = []            for w in words:                clean = w.strip(".,;:!?()[]{}\'\"-")                if clean and len(clean) >= cfg.min_word_length:                    fixed = en_spell.correction(clean)                    if fixed:                        w = w.replace(clean, fixed)                corrected.append(w)            return " ".join(corrected)        except Exception:            return text# Load correction rulescorrection_rules = load_correction_rules(cfg)calibrated_thresholds = auto_calibrate_dict_thresholds(correction_rules)print(f"Spell checking + Smart Dictionary loaded")print(f"  Correction rules: {len(correction_rules)}")print(f"  Calibrated thresholds: {calibrated_thresholds}")log_event("spell_init", {"rules_count": len(correction_rules)})

In [ ]:
# ========================================# الخلية 11: نظام نقاط الاستعادة (Checkpoint)# ========================================def save_checkpoint(data, path=None):    """Save checkpoint to JSON file."""    path = path or cfg.checkpoint_path    data["saved_at"] = datetime.now().isoformat()    with _db_lock:        with open(path, "w", encoding="utf-8") as f:            json.dump(data, f, ensure_ascii=False, indent=2)    logger.info(f"Checkpoint saved: {path}")def load_checkpoint(path=None):    """Load checkpoint from JSON file."""    path = path or cfg.checkpoint_path    if not os.path.exists(path):        return {}    try:        with open(path, "r", encoding="utf-8") as f:            return json.load(f)    except Exception as e:        logger.error(f"Failed to load checkpoint: {e}")        return {}def clear_checkpoint(path=None):    """Delete checkpoint file."""    path = path or cfg.checkpoint_path    if os.path.exists(path):        os.remove(path)print("Checkpoint system ready")

In [ ]:
# ========================================# الخلية 12: معالجة المستندات (صفحة بصفحة)# ========================================def load_pages(file_path, dpi=None, max_pages=None):    """Load PDF or image as list of numpy arrays. Memory-efficient."""    dpi = dpi or cfg.dpi    if file_path.lower().endswith(".pdf"):        images = convert_from_path(file_path, dpi=dpi, first_page=1, last_page=max_pages, thread_count=2)        return [np.array(img) for img in images]    else:        img = cv2.imread(file_path)        if img is None:            raise ValueError(f"Cannot read image: {file_path}")        return [img]def get_page_count(file_path):    """Get number of pages in PDF or 1 for images."""    if file_path.lower().endswith(".pdf"):        from pdf2image import pdfinfo_from_path        info = pdfinfo_from_path(file_path)        return info.get("Pages", 1)    return 1def smart_word_segmentation(image):    """Segment image into word regions."""    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]        kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 3))    dilated = cv2.dilate(thresh, kernel_h, iterations=1)        contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)        regions = []    img_h, img_w = gray.shape    for cnt in contours:        x, y, w, h = cv2.boundingRect(cnt)        if w > 10 and h > 10 and w < img_w * 0.9 and h < img_h * 0.5:            regions.append((x, y, w, h))        regions.sort(key=lambda r: (r[1], -r[0]))  # RTL: sort right-to-left within same line    return regionsdef process_document(file_path, run_id=None, page_range=None, progress_cb=None, resume=False):    """Full document processing pipeline (page-by-page for memory efficiency)."""    if not run_id:        run_id = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"        db.start_run(run_id, notes=f"File: {os.path.basename(file_path)}")    log_event("processing_start", {"run_id": run_id, "file": file_path})        total_pages = get_page_count(file_path)    start_page = (page_range[0] - 1) if page_range else 0    end_page = (page_range[1] - 1) if page_range else (total_pages - 1)    pages_to_process = end_page - start_page + 1        logger.info(f"Processing {os.path.basename(file_path)}: pages {start_page+1}-{end_page+1} of {total_pages}")        total_words = 0    start_time = time.time()        for page_idx in range(start_page, end_page + 1):        if progress_cb:            progress = (page_idx - start_page) / pages_to_process            progress_cb(progress, f"Processing page {page_idx + 1}/{total_pages}...")                if resume:            existing = db.get_words(page_num=page_idx + 1, run_id=run_id)            if existing:                total_words += len(existing)                continue                try:            pages = load_pages(file_path, max_pages=page_idx + 1)            if page_idx < len(pages):                page_img = pages[page_idx]            else:                break        except Exception as e:            logger.error(f"Failed to load page {page_idx + 1}: {e}")            continue                processed = preprocess_image(page_img)        regions = smart_word_segmentation(processed)                page_words = 0        for x, y, w, h in regions:            word_img = crop_safe(page_img, x, y, w, h)            word_img_proc = crop_safe(processed, x, y, w, h)                        result = ocr_engine.ensemble(word_img_proc)            raw_text = result["text"].strip()            confidence = result["confidence"]                        if not raw_text:                continue                        corrected, applied = apply_corrections(raw_text, correction_rules)            if corrected and corrected != raw_text:                corrected = correct_spelling(corrected)                        word_id = db.insert_word(                page_num=page_idx + 1, raw_text=raw_text,                confidence=confidence, ocr_engine=result["engine"],                word_bbox=json.dumps([x, y, w, h]),                top_predictions=json.dumps(result.get("all_candidates", {})),                run_id=run_id            )                        if corrected and corrected != raw_text:                db.update_word(word_id, corrected_text=corrected, status="auto_corrected")                        page_words += 1            total_words += 1                logger.info(f"  Page {page_idx + 1}: {page_words} words")        del page_img, processed        if torch.cuda.is_available():            torch.cuda.empty_cache()        elapsed = time.time() - start_time    stats = {        "run_id": run_id, "total_pages": pages_to_process, "total_words": total_words,        "elapsed_seconds": round(elapsed, 1),        "words_per_second": round(total_words / max(elapsed, 0.1), 2)    }    db.complete_run(run_id, total_pages=pages_to_process, total_words=total_words)    log_event("processing_complete", stats)        if progress_cb:        progress_cb(1.0, f"Done! {total_words} words in {elapsed:.1f}s")        if cfg.auto_export_after_run:        auto_export(run_id)        return statsprint("Document processing pipeline loaded")

In [ ]:
# ========================================# الخلية 13: نظام التصدير الشامل# ========================================def auto_export(run_id=None):    """Export data: CSV, XLSX, JSONL, TXT, PDF report."""    words = db.get_words(run_id=run_id) if run_id else db.get_words()    if not words:        logger.warning("No data to export")        return {}        export_dir = os.path.join(cfg.base_dir, "exports", run_id or "all")    os.makedirs(export_dir, exist_ok=True)        df = pd.DataFrame(words)    results = {}        # CSV    if cfg.export_csv:        csv_path = os.path.join(export_dir, "ocr_results.csv")        df.to_csv(csv_path, index=False, encoding="utf-8-sig")        results["csv"] = csv_path        # XLSX (per-page sheets)    if cfg.export_xlsx:        xlsx_path = os.path.join(export_dir, "ocr_results.xlsx")        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:            for page in sorted(df["page_num"].unique()):                page_df = df[df["page_num"] == page]                page_df.to_excel(writer, sheet_name=f"Page_{page}", index=False)            summary = db.count_by_status(run_id)            pd.DataFrame([summary]).to_excel(writer, sheet_name="Summary", index=False)        results["xlsx"] = xlsx_path        # JSONL    if cfg.export_jsonl:        jsonl_path = os.path.join(export_dir, "training_data.jsonl")        with open(jsonl_path, "w", encoding="utf-8") as f:            for w in words:                if w.get("corrected_text") and w.get("raw_text"):                    entry = {"raw_text": w["raw_text"], "corrected_text": w["corrected_text"],                             "confidence": w.get("confidence", 0), "page_num": w.get("page_num", 0)}                    f.write(json.dumps(entry, ensure_ascii=False) + "\n")        results["jsonl"] = jsonl_path        # TXT    if cfg.export_txt:        txt_path = os.path.join(export_dir, "full_text.txt")        with open(txt_path, "w", encoding="utf-8") as f:            for page in sorted(df["page_num"].unique()):                f.write(f"\n{'='*60}\nPage {page}\n{'='*60}\n\n")                page_words = df[df["page_num"] == page]                for _, w in page_words.iterrows():                    text = w.get("corrected_text") or w.get("raw_text", "")                    f.write(f"{text} ")                f.write("\n")        results["txt"] = txt_path        # PDF Report    if cfg.export_pdf_report:        try:            pdf_path = os.path.join(export_dir, "ocr_report.pdf")            generate_pdf_report(words, pdf_path)            results["pdf"] = pdf_path        except Exception as e:            logger.error(f"PDF export failed: {e}")        # Backup correction rules    rules_path = os.path.join(export_dir, "correction_rules.json")    with open(rules_path, "w", encoding="utf-8") as f:        rules_data = {k: asdict(v) for k, v in correction_rules.items()}        json.dump(rules_data, f, ensure_ascii=False, indent=2)    results["rules_json"] = rules_path        log_event("export", {"run_id": run_id, "formats": list(results.keys())})    logger.info(f"Exported: {list(results.keys())}")    return resultsdef generate_pdf_report(words, output_path):    """Generate PDF report."""    pdf = FPDF()    pdf.add_page()    pdf.set_font("Helvetica", size=12)    pdf.cell(0, 10, f"OCR Report - {datetime.now().strftime('%Y-%m-%d %H:%M')}", ln=True, align="C")    pdf.ln(5)    verified = sum(1 for w in words if w.get("status") == "verified")    pending = sum(1 for w in words if w.get("status") == "pending")    pdf.cell(0, 8, f"Total: {len(words)} | Verified: {verified} | Pending: {pending}", ln=True)    pdf.ln(5)    pdf.set_font("Helvetica", size=8)    for w in words[:100]:        raw = w.get("raw_text", "")[:30]        corr = w.get("corrected_text", "-")[:30]        conf = w.get("confidence", 0)        status = w.get("status", "")        pdf.cell(60, 6, raw, border=1)        pdf.cell(60, 6, corr, border=1)        pdf.cell(20, 6, f"{conf:.2f}", border=1)        pdf.cell(25, 6, status, border=1)        pdf.cell(0, 6, str(w.get("page_num", "")), border=1, ln=True)    pdf.output(output_path)def create_backup():    """Create timestamped backup."""    backup_dir = os.path.join(cfg.base_dir, f"backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}")    os.makedirs(backup_dir, exist_ok=True)    for name in ["handwriting.db", "feedback.csv", "correction_rules.json"]:        src = os.path.join(cfg.base_dir, name)        if os.path.exists(src):            shutil.copy2(src, os.path.join(backup_dir, name))    log_event("backup", {"path": backup_dir})    return backup_dirprint("Export system loaded (CSV, XLSX, JSONL, TXT, PDF)")

In [ ]:
# ========================================# الخلية 14: إعادة بناء الجمل# ========================================def reconstruct_sentences(words, y_threshold=30):    """Group words into sentences by page and Y-proximity."""    if not words:        return []        sentences = []    current_page = None    current_line_words = []    last_y = None        sorted_words = sorted(words, key=lambda w: (        w.get("page_num", 0),        json.loads(w.get("word_bbox", "[0,0,0,0]"))[1] if w.get("word_bbox") else 0    ))        for w in sorted_words:        page = w.get("page_num", 0)        bbox = json.loads(w.get("word_bbox", "[0,0,0,0]")) if w.get("word_bbox") else [0, 0, 0, 0]        y = bbox[1]                if page != current_page:            if current_line_words:                sentences.append(build_sentence(current_line_words))            current_line_words = [w]            current_page = page            last_y = y        elif last_y is not None and abs(y - last_y) > y_threshold:            sentences.append(build_sentence(current_line_words))            current_line_words = [w]            last_y = y        else:            current_line_words.append(w)            last_y = y        if current_line_words:        sentences.append(build_sentence(current_line_words))    return sentencesdef build_sentence(words):    """Build sentence dict from word records."""    raw = " ".join(w.get("raw_text", "") for w in words)    corrected = " ".join(w.get("corrected_text") or w.get("raw_text", "") for w in words)    any_corrected = any(        w.get("corrected_text") and w.get("corrected_text") != w.get("raw_text") for w in words    )    return {        "page_num": words[0].get("page_num", 0) if words else 0,        "raw_text": raw, "corrected_text": corrected,        "word_ids": [w.get("id") for w in words],        "all_corrected": any_corrected, "word_count": len(words), "status": "pending"    }print("Sentence reconstruction loaded")

In [ ]:
# ========================================# الخلية 15: مقاييس الأداء (WER/CER)# ========================================def compute_metrics(run_id=None):    """Compute WER and CER on verified data."""    verified = db.get_verified(run_id=run_id)    if len(verified) < 2:        return {"wer": 0.0, "cer": 0.0, "sample_count": len(verified)}        refs, hyps = [], []    for w in verified:        if w.get("raw_text") and w.get("corrected_text"):            refs.append(w["raw_text"])            hyps.append(w["corrected_text"])        if not refs:        return {"wer": 0.0, "cer": 0.0, "sample_count": 0}        try:        w = wer(refs, hyps)        c = cer(refs, hyps)    except Exception:        w, c = 0.0, 0.0        return {"wer": round(w, 4), "cer": round(c, 4), "sample_count": len(refs)}def plot_metrics_fig(run_id=None):    """Generate metrics visualization using matplotlib."""    import matplotlib    matplotlib.use("Agg")    import matplotlib.pyplot as plt        status_counts = db.count_by_status(run_id)    labels = list(status_counts.keys()) or ["No data"]    sizes = list(status_counts.values()) or [1]        fig, axes = plt.subplots(1, 2, figsize=(12, 5))        # Pie chart    colors = {"pending": "#FFA500", "verified": "#4CAF50", "auto_corrected": "#2196F3",              "rejected": "#F44336", "sentence_corrected": "#9C27B0"}    pie_colors = [colors.get(l, "#999") for l in labels]    axes[0].pie(sizes, labels=labels, autopct="%1.1f%%", colors=pie_colors, startangle=90)    axes[0].set_title("Word Status Distribution")        # Metrics bar    metrics = compute_metrics(run_id)    bars = axes[1].bar(["WER", "CER"], [metrics["wer"] * 100, metrics["cer"] * 100],                       color=["#FF6B6B", "#4ECDC4"])    axes[1].set_ylabel("Error Rate (%)")    axes[1].set_title(f"OCR Error Metrics (n={metrics['sample_count']})")    for bar, val in zip(bars, [metrics["wer"] * 100, metrics["cer"] * 100]):        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,                    f"{val:.1f}%", ha="center", fontweight="bold")        plt.tight_layout()        chart_path = os.path.join(cfg.base_dir, "metrics_chart.png")    plt.savefig(chart_path, dpi=100, bbox_inches="tight")    plt.close()    return chart_pathprint("Metrics system loaded (WER/CER)")

In [ ]:
# ========================================# الخلية 16: التعلم النشط (Active Learning)# ========================================def count_corrections_since_last_training():    """Count new corrections since last training run."""    events = db.get_review_events(limit=500)    train_events = [e for e in events if e["action"] == "confirm" or e["action"] == "save"]    return len(train_events)def should_trigger_training():    """Determine if enough new data to trigger fine-tuning."""    return count_corrections_since_last_training() >= cfg.al_min_correctionsdef reprocess_low_confidence(run_id=None):    """Re-process words with low confidence using updated models."""    low_conf = db.get_low_confidence(threshold=cfg.al_low_conf_threshold, run_id=run_id)    count = 0    for word in low_conf:        # Re-OCR would require storing original images        # For now, mark for review        count += 1    logger.info(f"Found {count} low-confidence words for re-processing")    return countdef active_learning_cycle():    """Run one active learning cycle: check data -> train -> reprocess."""    if not should_trigger_training():        return {"action": "skip", "reason": "Not enough new corrections"}        corrections = count_corrections_since_last_training()    logger.info(f"Active Learning: {corrections} new corrections, triggering training")        # Export training data    auto_export("al_training")        return {"action": "train", "corrections": corrections}print("Active Learning system loaded")

In [ ]:
# ========================================# الخلية 17: ضبط نماذج TrOCR بـ LoRA# ========================================def prepare_training_data(run_id=None, max_samples=None):    """Prepare training dataset from verified words. Returns image-text pairs."""    max_samples = max_samples or cfg.ft_max_samples    verified = db.get_verified(run_id=run_id, limit=max_samples)        if not verified:        logger.warning("No verified data for training")        return [], []        # We need the original images to create training data    # For notebook-based training, we use the text pairs for language model fine-tuning    texts = [w["corrected_text"] or w["raw_text"] for w in verified if w.get("raw_text")]    labels = [w["raw_text"] for w in verified if w.get("raw_text")]        return texts, labelsdef finetune_trocr_lora(output_dir=None, run_id=None, progress_cb=None):    """LoRA fine-tuning of TrOCR model with TensorBoard logging."""    import matplotlib    matplotlib.use("Agg")        output_dir = output_dir or os.path.join(cfg.base_dir, "finetuned_model")        texts, labels = prepare_training_data(run_id=run_id)    if len(texts) < 5:        msg = f"Not enough training data: {len(texts)} samples (need at least 5)"        logger.warning(msg)        return {"success": False, "message": msg}        logger.info(f"Starting LoRA fine-tuning with {len(texts)} samples")    if progress_cb:        progress_cb(0.1, f"Preparing {len(texts)} training samples...")        # Configure LoRA    lora_config = LoraConfig(        r=cfg.lora_r,        lora_alpha=cfg.lora_alpha,        lora_dropout=cfg.lora_dropout,        target_modules=cfg.lora_target_modules,        task_type=TaskType.SEQ_2_SEQ_LM,        bias="none"    )        model = ocr_engine.trocr_model    model = get_peft_model(model, lora_config)    model.print_trainable_parameters()        if progress_cb:        progress_cb(0.3, "LoRA configured, setting up training...")        # Create simple text-based training    tokenizer = ocr_engine.trocr_processor.tokenizer        class OCRDataset(torch.utils.data.Dataset):        def __init__(self, texts, labels, processor):            self.texts = texts            self.labels = labels            self.processor = processor                def __len__(self):            return len(self.texts)                def __getitem__(self, idx):            text = self.texts[idx]            label = self.labels[idx]            encoding = self.processor(                text=label, text_pair=text,                 return_tensors="pt", padding="max_length",                max_length=128, truncation=True            )            encoding = {k: v.squeeze(0) for k, v in encoding.items()}            encoding["labels"] = encoding["input_ids"].clone()            return encoding        dataset = OCRDataset(texts, labels, ocr_engine.trocr_processor)        training_args = TrainingArguments(        output_dir=output_dir,        num_train_epochs=cfg.ft_epochs,        per_device_train_batch_size=cfg.ft_batch_size,        learning_rate=cfg.ft_learning_rate,        logging_steps=10,        save_strategy="epoch",        remove_unused_columns=False,        report_to="tensorboard",        dataloader_pin_memory=False,    )        trainer = Trainer(        model=model,        args=training_args,        train_dataset=dataset,        data_collator=default_data_collator,    )        if progress_cb:        progress_cb(0.5, "Training started...")        trainer.train()        if progress_cb:        progress_cb(0.9, "Saving model...")        model.save_pretrained(output_dir)    ocr_engine.trocr_processor.save_pretrained(output_dir)        log_event("finetune_complete", {        "samples": len(texts), "epochs": cfg.ft_epochs,        "output_dir": output_dir    })        if progress_cb:        progress_cb(1.0, f"Fine-tuning complete! Model saved to {output_dir}")        return {"success": True, "samples": len(texts), "output_dir": output_dir}print("LoRA Fine-tuning system loaded")

In [ ]:
# ========================================# الخلية 18: دليل الدراسة + مخططات Mermaid# ========================================def generate_study_guide(run_id=None, output_format="html"):    """Generate study guide with terminology tables, Mermaid diagrams, and flashcards."""    words = db.get_words(run_id=run_id)    if not words:        return "No data available for study guide."        df = pd.DataFrame(words)        # Extract unique terms    all_terms = set()    for _, w in df.iterrows():        text = w.get("corrected_text") or w.get("raw_text", "")        for word in text.split():            if len(word) >= 3:                all_terms.add(word.strip(".,;:!?()"))        # Build terminology table    terms_html = ""    for i, term in enumerate(sorted(all_terms)[:50], 1):        count = sum(1 for w in words if term in (w.get("corrected_text") or w.get("raw_text", "")))        terms_html += f"<tr><td>{i}</td><td>{term}</td><td>{count}</td></tr>"        # Mermaid diagram    mermaid_code = "graph LR\n"    unique_pages = sorted(df["page_num"].unique())[:10]    for p in unique_pages:        page_words = df[df["page_num"] == p]        count = len(page_words)        verified = sum(1 for _, w in page_words.iterrows() if w.get("status") == "verified")        mermaid_code += f'    P{p}["Page {p}: {count} words ({verified} verified)"]\n'        for i in range(len(unique_pages) - 1):        mermaid_code += f"    P{unique_pages[i]} --> P{unique_pages[i+1]}\n"        # Flashcards    flashcards = ""    for _, w in df.head(20).iterrows():        raw = w.get("raw_text", "")        corr = w.get("corrected_text", "") or raw        if raw != corr:            flashcards += f'<div class="flashcard"><div class="front">{raw}</div><div class="back">{corr}</div></div>'        # Generate HTML    html_content = f"""<!DOCTYPE html><html dir="rtl" lang="ar"><head>    <meta charset="UTF-8">    <title>Study Guide - OmniFile OCR</title>    <style>        body {{ font-family: 'Amiri', 'Arial', sans-serif; direction: rtl; margin: 20px; background: #f5f5f5; }}        h1 {{ color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 10px; }}        h2 {{ color: #2980b9; margin-top: 30px; }}        table {{ border-collapse: collapse; width: 100%; background: white; margin: 15px 0; }}        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: right; }}        th {{ background: #3498db; color: white; }}        tr:nth-child(even) {{ background: #f2f2f2; }}        .flashcard {{ background: white; border: 2px solid #3498db; border-radius: 10px;                       padding: 15px; margin: 10px; display: inline-block; min-width: 150px; text-align: center; }}        .front {{ font-size: 18px; color: #e74c3c; font-weight: bold; }}        .back {{ font-size: 16px; color: #27ae60; margin-top: 5px; }}        .mermaid {{ background: white; padding: 20px; border-radius: 10px; margin: 15px 0; }}        pre {{ background: #2c3e50; color: #ecf0f1; padding: 15px; border-radius: 5px; overflow-x: auto; }}        .stats {{ background: white; padding: 20px; border-radius: 10px; display: flex; gap: 20px; flex-wrap: wrap; }}        .stat-box {{ background: #ecf0f1; padding: 15px; border-radius: 8px; text-align: center; min-width: 120px; }}        .stat-num {{ font-size: 28px; font-weight: bold; color: #2c3e50; }}        @media print {{ body {{ background: white; }} .flashcard {{ break-inside: avoid; }} }}    </style></head><body>    <h1>Study Guide - {datetime.now().strftime("%Y-%m-%d")}</h1>        <div class="stats">        <div class="stat-box">            <div class="stat-num">{len(words)}</div>            <div>Total Words</div>        </div>        <div class="stat-box">            <div class="stat-num">{df["page_num"].nunique()}</div>            <div>Pages</div>        </div>        <div class="stat-box">            <div class="stat-num">{len(all_terms)}</div>            <div>Unique Terms</div>        </div>        <div class="stat-box">            <div class="stat-num">{sum(1 for w in words if w.get("status")=="verified")}</div>            <div>Verified</div>        </div>    </div>        <h2>Terminology Table</h2>    <table>        <tr><th>#</th><th>Term</th><th>Frequency</th></tr>        {terms_html}    </table>        <h2>Document Structure (Mermaid)</h2>    <div class="mermaid">        <pre>{mermaid_code}</pre>    </div>        <h2>Correction Flashcards</h2>    <div>        {flashcards if flashcards else "<p>No corrections to show as flashcards.</p>"}    </div>        <h2>Metrics</h2>    <p>WER: {compute_metrics().get("wer", 0):.2%} | CER: {compute_metrics().get("cer", 0):.2%}</p></body></html>"""        output_path = os.path.join(cfg.base_dir, "exports", run_id or "all", "study_guide.html")    os.makedirs(os.path.dirname(output_path), exist_ok=True)    with open(output_path, "w", encoding="utf-8") as f:        f.write(html_content)        log_event("study_guide", {"run_id": run_id, "terms": len(all_terms)})    return output_pathprint("Study Guide system loaded")

In [ ]:
# ========================================# الخلية 19: استخراج المفردات ثنائية اللغة# ========================================def extract_bilingual_vocab(run_id=None):    """Extract English-Arabic vocabulary pairs from verified data."""    words = db.get_verified(run_id=run_id)    if not words:        return pd.DataFrame(columns=["english", "arabic", "page", "confidence"])        pairs = []    for w in words:        text = w.get("corrected_text") or w.get("raw_text", "")        # Detect language per word        for word in text.split():            clean = word.strip(".,;:!?()[]{}\'\\"")            if not clean:                continue            # Check if Arabic (Unicode range)            is_arabic = any("\u0600" <= c <= "\u06FF" for c in clean)            is_english = any("a" <= c.lower() <= "z" for c in clean)                        if is_arabic and not is_english:                pairs.append({"arabic": clean, "english": "", "page": w.get("page_num", 0),                             "confidence": w.get("confidence", 0)})            elif is_english and not is_arabic:                pairs.append({"english": clean, "arabic": "", "page": w.get("page_num", 0),                             "confidence": w.get("confidence", 0)})        df = pd.DataFrame(pairs)        if not df.empty:        output_path = os.path.join(cfg.base_dir, "exports", run_id or "all", "bilingual_vocab.csv")        os.makedirs(os.path.dirname(output_path), exist_ok=True)        df.to_csv(output_path, index=False, encoding="utf-8-sig")        logger.info(f"Bilingual vocab extracted: {len(df)} words")        return dfprint("Bilingual vocab extraction loaded")

In [ ]:
# ========================================# الخلية 20: واجهة Gradio الرئيسية (8 تبويبات)# ========================================# Review state_word_review_state = {"idx": 0, "words": [], "run_id": None}_sent_review_state = {"idx": 0, "sentences": [], "run_id": None}_undo_stack = []def _load_word_reviews(run_id):    words = db.get_words(run_id=run_id, status="pending") + db.get_words(run_id=run_id, status="auto_corrected")    if not words:        words = db.get_words(status="pending", limit=200)    _word_review_state["words"] = words    _word_review_state["idx"] = 0    _word_review_state["run_id"] = run_id    return wordsdef _get_current_word():    idx = _word_review_state["idx"]    words = _word_review_state["words"]    if not words or idx >= len(words):        return None    return words[idx]def word_show_current(run_id):    _load_word_reviews(run_id)    w = _get_current_word()    if not w:        return "No pending words to review", "", 0.0, ""        raw = w.get("raw_text", "")    corr = w.get("corrected_text", "") or ""    conf = w.get("confidence", 0)        cands = ""    try:        preds = json.loads(w.get("top_predictions", "{}")) if isinstance(w.get("top_predictions"), str) else {}        if preds:            cands = " | ".join(f"{t} ({c:.0%})" for t, c in sorted(preds.items(), key=lambda x: -x[1])[:5])    except Exception:        pass        idx_info = f"Word {_word_review_state['idx']+1} of {len(_word_review_state['words'])}"    return f"{idx_info}\n\nRaw: {raw}\nEngine: {w.get('ocr_engine', '')}", corr, conf, candsdef word_confirm(text, run_id):    w = _get_current_word()    if not w:        return word_show_current(run_id)        old = w.get("corrected_text", "")    new = text.strip() if text.strip() else w.get("raw_text", "")        _undo_stack.append({"word_id": w["id"], "old_corrected": old, "old_status": w.get("status", "pending")})        db.update_word(w["id"], corrected_text=new, status="verified")    db.log_review(run_id, w["id"], "confirm", old_value=old, new_value=new)        _word_review_state["idx"] += 1    return word_show_current(run_id)def word_delete(run_id):    w = _get_current_word()    if not w:        return word_show_current(run_id)        _undo_stack.append({"word_id": w["id"], "old_corrected": w.get("corrected_text", ""), "old_status": w.get("status", "pending")})    db.update_word(w["id"], status="rejected")    db.log_review(run_id, w["id"], "delete")        _word_review_state["idx"] += 1    return word_show_current(run_id)def word_next(run_id):    _word_review_state["idx"] += 1    return word_show_current(run_id)def word_prev(run_id):    if _word_review_state["idx"] > 0:        _word_review_state["idx"] -= 1    return word_show_current(run_id)def word_copy_raw(run_id):    w = _get_current_word()    if not w:        return ""    return w.get("raw_text", "")def word_undo(run_id):    if not _undo_stack:        return word_show_current(run_id)        action = _undo_stack.pop()    db.update_word(action["word_id"], corrected_text=action["old_corrected"], status=action["old_status"])        if _word_review_state["idx"] > 0:        _word_review_state["idx"] -= 1        return word_show_current(run_id)# --- Sentence Review ---def _load_sent_reviews(run_id):    all_words = db.get_words(run_id=run_id)    sents = reconstruct_sentences(all_words)    _sent_review_state["sentences"] = sents    _sent_review_state["idx"] = 0    _sent_review_state["run_id"] = run_id    return sentsdef sent_show_current(run_id):    _load_sent_reviews(run_id)    idx = _sent_review_state["idx"]    sents = _sent_review_state["sentences"]    if not sents or idx >= len(sents):        return "No sentences to review", ""    s = sents[idx]    info = f"Sentence {idx+1} of {len(sents)} (Page {s['page_num']}, {s['word_count']} words)"    return f"{info}\n\n{s['raw_text']}", s["corrected_text"]def sent_save(text, run_id):    idx = _sent_review_state["idx"]    sents = _sent_review_state["sentences"]    if not sents or idx >= len(sents):        return sent_show_current(run_id)    s = sents[idx]    for wid in s["word_ids"]:        db.update_word(wid, status="sentence_corrected")        db.log_review(run_id, wid, "save_sentence")    _sent_review_state["idx"] += 1    return sent_show_current(run_id)def sent_next(run_id):    _sent_review_state["idx"] += 1    return sent_show_current(run_id)def sent_prev(run_id):    if _sent_review_state["idx"] > 0:        _sent_review_state["idx"] -= 1    return sent_show_current(run_id)# --- Dashboard ---def get_dashboard_text(run_id):    counts = db.count_by_status(run_id)    total = sum(counts.values())    if total == 0:        return "No data yet. Process a document first."        verified = counts.get("verified", 0)    pending = counts.get("pending", 0)    auto = counts.get("auto_corrected", 0)        pct = f"{verified/total:.1%}" if total else "0%"        metrics = compute_metrics(run_id)    runs = db.get_run_history(limit=5)        text = f"## Dashboard\n\n"    text += f"**Total Words:** {total}\n\n"    text += f"| Status | Count | Percentage |\n"    text += f"|--------|-------|------------|\n"    for status, count in counts.items():        text += f"| {status} | {count} | {count/total:.1%} |\n"    text += f"\n**Verification Rate:** {pct}\n\n"    text += f"**WER:** {metrics['wer']:.2%} | **CER:** {metrics['cer']:.2%}\n\n"    text += f"**Run History:**\n"    for r in runs:        status_icon = "OK" if r["status"] == "completed" else "..."        text += f"- {r['run_id']}: {r.get('total_words', 0)} words, {status_icon}\n"        return textdef get_dashboard_chart(run_id):    try:        chart_path = plot_metrics_fig(run_id)        return chart_path    except Exception as e:        return None# --- Dictionary Review ---def get_dict_review_text(filter_status="all"):    if not correction_rules:        return "No correction rules loaded."        text = "## Smart Dictionary Review\n\n"    text += f"Total rules: {len(correction_rules)}\n\n"        filtered = correction_rules    if filter_status != "all":        filtered = {k: v for k, v in correction_rules.items() if v.status == filter_status}        text += f"| # | Original | Correction | Confidence | Usage | Status | Indicator |\n"    text += f"|---|----------|------------|------------|-------|--------|-----------|\n"        for i, (orig, rule) in enumerate(sorted(filtered.items())[:50], 1):        indicator = calculate_rule_indicator(rule)        text += f"| {i} | {orig} | {rule.correction} | {rule.confidence:.1%} | {rule.usage_count} | {rule.status} | {indicator} |\n"        return textdef dict_calibrate():    global calibrated_thresholds    calibrated_thresholds = auto_calibrate_dict_thresholds(correction_rules)    return f"Calibrated thresholds: {json.dumps(calibrated_thresholds, indent=2)}"def dict_save_to_file():    path = os.path.join(cfg.base_dir, "correction_rules.json")    with open(path, "w", encoding="utf-8") as f:        rules_data = {k: asdict(v) for k, v in correction_rules.items()}        json.dump(rules_data, f, ensure_ascii=False, indent=2)    return f"Saved {len(correction_rules)} rules to {path}"# --- Processing ---def do_process(file_path, page_start, page_end, resume, progress=gr.Progress()):    if not file_path:        return "Please enter a file path."        try:        page_range = None        if page_start and page_end:            page_range = (int(page_start), int(page_end))                stats = process_document(            file_path,            page_range=page_range,            resume=resume,            progress_cb=lambda p, m: progress(p, m)        )        return f"Processing complete!\n\n{json.dumps(stats, indent=2, default=str)}"    except Exception as e:        return f"Error: {e}"def do_export(run_id):    try:        results = auto_export(run_id if run_id.strip() else None)        if not results:            return "No data to export."        return f"Exported formats: {', '.join(results.keys())}\n\n" + "\n".join(f"- {fmt}: {path}" for fmt, path in results.items())    except Exception as e:        return f"Export error: {e}"def do_backup():    try:        path = create_backup()        return f"Backup created at: {path}"    except Exception as e:        return f"Backup error: {e}"def do_metrics(run_id):    try:        metrics = compute_metrics(run_id if run_id.strip() else None)        chart = get_dashboard_chart(run_id if run_id.strip() else None)        text = f"## Metrics\n\n**WER:** {metrics['wer']:.2%}\n\n**CER:** {metrics['cer']:.2%}\n\n**Samples:** {metrics['sample_count']}"        return text, chart    except Exception as e:        return f"Error computing metrics: {e}", Nonedef do_finetune(run_id, progress=gr.Progress()):    try:        progress(0, "Starting fine-tuning...")        result = finetune_trocr_lora(            run_id=run_id if run_id.strip() else None,            progress_cb=lambda p, m: progress(p, m)        )        return json.dumps(result, indent=2, default=str)    except Exception as e:        return f"Fine-tuning error: {e}"def do_study_guide(run_id):    try:        path = generate_study_guide(run_id if run_id.strip() else None)        return f"Study guide generated: {path}"    except Exception as e:        return f"Error: {e}"def do_migration():    try:        found = migrator.scan()        if not found:            return "No old projects found."        report = migrator.migrate(delete_after=False)        return f"Migration complete!\n\nFound: {list(found.keys())}\nImported: {report['imported']} words"    except Exception as e:        return f"Migration error: {e}"def do_bilingual(run_id):    try:        df = extract_bilingual_vocab(run_id if run_id.strip() else None)        if df.empty:            return "No bilingual vocab found."        return f"Extracted {len(df)} terms\n\n" + df.head(30).to_string()    except Exception as e:        return f"Error: {e}"print("All Gradio callback functions loaded")

In [ ]:
# ========================================# الخلية 21: بناء واجهة Gradio وإطلاقها# ========================================def build_app():    with gr.Blocks(        title="OmniFile Arabic OCR",        theme=gr.themes.Soft(),        css=".gradio-container { max-width: 1200px; margin: auto; }"    ) as app:        gr.Markdown("# OmniFile Arabic OCR - Unified\n### Arabic Handwriting Recognition + Spell Correction + AI Enhancement")                with gr.Tabs():            # === Tab 1: Processing ===            with gr.Tab("Processing"):                with gr.Row():                    with gr.Column():                        file_input = gr.Textbox(label="File Path (PDF or Image)", placeholder="/content/drive/MyDrive/document.pdf")                        with gr.Row():                            page_start = gr.Number(label="Start Page", value=1, precision=0)                            page_end = gr.Number(label="End Page", value=None, precision=0)                        resume_cb = gr.Checkbox(label="Resume (skip already processed pages)", value=False)                        process_btn = gr.Button("Start Processing", variant="primary")                    with gr.Column():                        process_output = gr.Textbox(label="Results", lines=10)                                process_btn.click(                    do_process,                    inputs=[file_input, page_start, page_end, resume_cb],                    outputs=[process_output]                )                        # === Tab 2: Word Review ===            with gr.Tab("Word Review"):                run_id_word = gr.Textbox(label="Run ID (optional)", placeholder="run_YYYYMMDD_HHMMSS")                with gr.Row():                    word_info = gr.Textbox(label="Word Info", lines=4)                    word_candidates = gr.Textbox(label="OCR Candidates", lines=2)                with gr.Row():                    word_edit = gr.Textbox(label="Corrected Text")                    word_conf = gr.Number(label="Confidence")                with gr.Row():                    btn_copy_raw = gr.Button("Copy Raw")                    btn_undo = gr.Button("Undo")                    btn_prev = gr.Button("Previous")                    btn_confirm = gr.Button("Confirm", variant="primary")                    btn_delete = gr.Button("Delete")                    btn_next = gr.Button("Next")                                word_show_btn = gr.Button("Load Words", variant="secondary")                word_show_btn.click(word_show_current, inputs=[run_id_word], outputs=[word_info, word_edit, word_conf, word_candidates])                                btn_copy_raw.click(word_copy_raw, inputs=[run_id_word], outputs=[word_edit])                btn_confirm.click(word_confirm, inputs=[word_edit, run_id_word], outputs=[word_info, word_edit, word_conf, word_candidates])                btn_delete.click(word_delete, inputs=[run_id_word], outputs=[word_info, word_edit, word_conf, word_candidates])                btn_next.click(word_next, inputs=[run_id_word], outputs=[word_info, word_edit, word_conf, word_candidates])                btn_prev.click(word_prev, inputs=[run_id_word], outputs=[word_info, word_edit, word_conf, word_candidates])                btn_undo.click(word_undo, inputs=[run_id_word], outputs=[word_info, word_edit, word_conf, word_candidates])                        # === Tab 3: Sentence Review ===            with gr.Tab("Sentence Review"):                run_id_sent = gr.Textbox(label="Run ID (optional)")                sent_info = gr.Textbox(label="Sentence", lines=3)                sent_edit = gr.Textbox(label="Corrected Sentence", lines=3)                with gr.Row():                    btn_sent_prev = gr.Button("Previous")                    btn_sent_save = gr.Button("Save", variant="primary")                    btn_sent_next = gr.Button("Next")                                sent_show_btn = gr.Button("Load Sentences", variant="secondary")                sent_show_btn.click(sent_show_current, inputs=[run_id_sent], outputs=[sent_info, sent_edit])                btn_sent_save.click(sent_save, inputs=[sent_edit, run_id_sent], outputs=[sent_info, sent_edit])                btn_sent_next.click(sent_next, inputs=[run_id_sent], outputs=[sent_info, sent_edit])                btn_sent_prev.click(sent_prev, inputs=[run_id_sent], outputs=[sent_info, sent_edit])                        # === Tab 4: Dashboard ===            with gr.Tab("Dashboard"):                run_id_dash = gr.Textbox(label="Run ID (optional)")                with gr.Row():                    dash_refresh = gr.Button("Refresh Dashboard", variant="secondary")                    metrics_btn = gr.Button("Compute Metrics", variant="primary")                    export_btn = gr.Button("Export All Formats")                    backup_btn = gr.Button("Create Backup")                dash_text = gr.Markdown("Click 'Refresh Dashboard' to load stats.")                dash_chart = gr.Image(label="Charts")                                dash_refresh.click(get_dashboard_text, inputs=[run_id_dash], outputs=[dash_text])                metrics_btn.click(do_metrics, inputs=[run_id_dash], outputs=[dash_text, dash_chart])                export_btn.click(do_export, inputs=[run_id_dash], outputs=[dash_text])                backup_btn.click(do_backup, outputs=[dash_text])                        # === Tab 5: Fine-tuning & Active Learning ===            with gr.Tab("Fine-tuning"):                run_id_ft = gr.Textbox(label="Run ID (optional)")                ft_btn = gr.Button("Start LoRA Fine-tuning", variant="primary")                al_btn = gr.Button("Check Active Learning")                ft_output = gr.Textbox(label="Output", lines=8)                                ft_btn.click(do_finetune, inputs=[run_id_ft], outputs=[ft_output])                al_btn.click(lambda rid: json.dumps(active_learning_cycle(), indent=2, default=str), inputs=[run_id_ft], outputs=[ft_output])                        # === Tab 6: Study Guide ===            with gr.Tab("Study Guide"):                run_id_sg = gr.Textbox(label="Run ID (optional)")                sg_btn = gr.Button("Generate Study Guide", variant="primary")                sg_output = gr.Textbox(label="Output", lines=5)                                sg_btn.click(do_study_guide, inputs=[run_id_sg], outputs=[sg_output])                        # === Tab 7: Smart Dictionary ===            with gr.Tab("Smart Dictionary"):                dict_filter = gr.Dropdown(["all", "pending", "approved", "rejected"], label="Filter by Status", value="all")                with gr.Row():                    dict_refresh = gr.Button("Refresh Dictionary")                    dict_cal_btn = gr.Button("Auto-Calibrate Thresholds")                    dict_save_btn = gr.Button("Save to File")                dict_text = gr.Markdown("Click 'Refresh Dictionary' to load rules.")                                dict_refresh.click(get_dict_review_text, inputs=[dict_filter], outputs=[dict_text])                dict_cal_btn.click(dict_calibrate, outputs=[dict_text])                dict_save_btn.click(dict_save_to_file, outputs=[dict_text])                        # === Tab 8: Settings & Migration ===            with gr.Tab("Settings"):                with gr.Row():                    migrate_btn = gr.Button("Scan & Migrate Old Projects")                    bilingual_btn = gr.Button("Extract Bilingual Vocab")                    health_btn = gr.Button("Health Check")                settings_output = gr.Textbox(label="Output", lines=10)                                migrate_btn.click(do_migration, outputs=[settings_output])                bilingual_btn.click(do_bilingual, inputs=[run_id_dash], outputs=[settings_output])                health_btn.click(lambda: json.dumps(write_health_snapshot(), indent=2, default=str), outputs=[settings_output])                return app# Build and launchapp = build_app()print("Gradio app built successfully with 8 tabs")# Auto-launch in Colabif IN_COLAB:    app.launch(        share=cfg.gradio_share,        server_name=cfg.gradio_server_name,        server_port=cfg.gradio_server_port,        debug=cfg.gradio_debug    )else:    print("Run app.launch() manually, or execute the next cell.")

In [ ]:
# ========================================# الخلية 22: إطلاق يدوي (لغير Colab)# ========================================# Uncomment the line below to launch manually:# app.launch(share=True, server_name="0.0.0.0")# Or with specific port:# app.launch(share=True, server_name="0.0.0.0", server_port=7860)